In [15]:
# ============================================================
# CodPerformancePanel.ipynb  [VERSIONE OTTIMIZZATA RAM]
# ============================================================
# Script AUTONOMO — non modifica CodMatchFreddieHMDA.
#
# INPUT:
#   1. matched_{YEAR}.csv  output di CodMatchFreddieHMDA
#      (una riga per loan, con dati origination Freddie + HMDA)
#   2. Stessi zip Freddie Mac (stessa struttura del notebook originale)
#      I file TIME (performance) sono estratti dagli stessi zip.
#
# OUTPUT:
#   panel_{YEAR}.csv
#   Una riga per (loan_sequence_number, mese)
#
# STRUTTURA DRIVE ATTESA:
#   MyDrive/
#     thesis_data/
#       freddie/
#         historical_data_{YEAR}/
#           historical_data_{YEAR}Q1.zip
#           historical_data_{YEAR}Q2.zip
#           ...
#       output/
#         matched_{YEAR}.csv              <- input
#         panel_{YEAR}.csv               <- output
# ============================================================

!pip install pyarrow -q

from google.colab import drive
drive.mount('/content/drive')

import os
print('Drive montato:', os.path.exists('/content/drive/MyDrive'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive montato: True


In [16]:
# ============================================================
# CELLA 1 - Configurazione
# ============================================================
# MODIFICA SOLO DRIVE_ROOT e YEAR.

import os, glob, zipfile, shutil, gc
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

# ---- UNICI DUE VALORI DA MODIFICARE ----
DRIVE_ROOT = '/content/drive/MyDrive/thesis_data'
YEAR       = 2018
# ----------------------------------------

FREDDIE_DIR  = os.path.join(DRIVE_ROOT, 'freddie')
OUTPUT_DIR   = os.path.join(DRIVE_ROOT, 'output')
PERF_LOCAL   = '/content/freddie_perf_local'
PERF_PARQUET = '/content/perf_tmp.parquet'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PERF_LOCAL, exist_ok=True)

MATCHED_PATH = os.path.join(OUTPUT_DIR, f'matched_{YEAR}.csv')
OUTPUT_PATH  = os.path.join(OUTPUT_DIR, f'panel_{YEAR}.csv')

print(f'Anno             : {YEAR}')
print(f'Freddie zip dir  : {FREDDIE_DIR}')
print(f'File matched     : {MATCHED_PATH}')
print(f'Output panel     : {OUTPUT_PATH}')
print()
for label, path in [('Freddie zip dir', FREDDIE_DIR), ('matched CSV', MATCHED_PATH)]:
    status = 'OK' if os.path.exists(path) else 'ERRORE - non trovato'
    print(f'  {status}  {label}: {path}')

Anno             : 2018
Freddie zip dir  : /content/drive/MyDrive/thesis_data/freddie
File matched     : /content/drive/MyDrive/thesis_data/output/matched_2018.csv
Output panel     : /content/drive/MyDrive/thesis_data/output/panel_2018.csv

  OK  Freddie zip dir: /content/drive/MyDrive/thesis_data/freddie
  OK  matched CSV: /content/drive/MyDrive/thesis_data/output/matched_2018.csv


In [17]:
# ============================================================
# CELLA 2 - Layout colonne file performance
# ============================================================

PERF_COLS_FULL = [
    'loan_sequence_number',
    'monthly_reporting_period',
    'current_upb',
    'current_loan_delinquency_status',
    'loan_age',
    'remaining_months_to_maturity',
    'defect_settlement_date',
    'modifications_flag',
    'zero_balance_code',
    'zero_balance_effective_date',
    'current_interest_rate',
    'current_deferred_upb',
    'due_date_last_paid_installment',
    'mi_recoveries',
    'net_sales_proceeds',
    'non_mi_recoveries',
    'expenses',
    'legal_costs',
    'maintenance_costs',
    'taxes_and_insurance',
    'miscellaneous_expenses',
    'actual_loss',
    'modification_cost',
    'step_modification_flag',
    'deferred_payment_plan',
    'estimated_ltv',
    'zero_balance_removal_upb',
    'delinquent_accrued_interest',
    'delinquency_due_to_disaster',
    'borrower_assistance_status',
    'current_month_modification_loss',
    'interest_bearing_upb',
]

PERF_COLS_KEEP = [
    'loan_sequence_number',
    'monthly_reporting_period',
    'current_upb',
    'current_loan_delinquency_status',
    'loan_age',
    'remaining_months_to_maturity',
    'modifications_flag',
    'zero_balance_code',
    'zero_balance_effective_date',
    'current_interest_rate',
    'current_deferred_upb',
    'estimated_ltv',
    'delinquency_due_to_disaster',
    'borrower_assistance_status',
]

# Schema Parquet fisso: tutte string, evita mismatch tra chunk
PARQUET_SCHEMA = pa.schema([(col, pa.string()) for col in PERF_COLS_KEEP])

print(f'Colonne nel manuale  : {len(PERF_COLS_FULL)}')
print(f'Colonne da mantenere : {len(PERF_COLS_KEEP)}')

Colonne nel manuale  : 32
Colonne da mantenere : 14


In [18]:
# ============================================================
# CELLA 3 - Estrazione file TIME dagli zip Freddie
# ============================================================

def extract_performance_zips(year):
    zip_pattern_sub  = os.path.join(FREDDIE_DIR, f'historical_data_{year}',
                                    f'historical_data_{year}Q*.zip')
    zip_pattern_flat = os.path.join(FREDDIE_DIR, f'historical_data_{year}Q*.zip')
    zip_files = glob.glob(zip_pattern_sub) or glob.glob(zip_pattern_flat)

    if not zip_files:
        raise FileNotFoundError(
            f'Nessun zip trovato per {year}.\n'
            f'Pattern cercati:\n  {zip_pattern_sub}\n  {zip_pattern_flat}'
        )

    print(f'Trovati {len(zip_files)} zip per {year}:')
    extracted = []

    for zpath in sorted(zip_files):
        zname = os.path.basename(zpath)
        with zipfile.ZipFile(zpath, 'r') as z:
            contents = z.namelist()
            time_files = [f for f in contents
                          if 'time' in f.lower() and f.endswith('.txt')]

            if not time_files:
                print(f'  ATTENZIONE: nessun file TIME in {zname}')
                print(f'  Contenuto zip: {contents}')
                continue

            for tf in time_files:
                out_name = os.path.basename(tf)
                dest = os.path.join(PERF_LOCAL, out_name)

                if os.path.exists(dest):
                    size_mb = os.path.getsize(dest) / 1e6
                    print(f'  {zname} -> {out_name} gia estratto ({size_mb:.0f} MB), skip')
                else:
                    print(f'  Estraggo {out_name} da {zname}...', end=' ', flush=True)
                    with z.open(tf) as src, open(dest, 'wb') as dst:
                        shutil.copyfileobj(src, dst)
                    size_mb = os.path.getsize(dest) / 1e6
                    print(f'OK ({size_mb:.0f} MB)')

                extracted.append(dest)

    if not extracted:
        raise RuntimeError(
            'Nessun file TIME estratto. Verifica che gli zip contengano '
            'file con "time" nel nome (es. time_2024Q1.txt).'
        )
    return sorted(extracted)


print(f'Estraggo file performance per {YEAR}...')
perf_txt_files = extract_performance_zips(YEAR)
print(f'\nFile TIME estratti: {len(perf_txt_files)}')
for f in perf_txt_files:
    print(f'  {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.0f} MB)')

Estraggo file performance per 2018...
Trovati 4 zip per 2018:
  Estraggo historical_data_time_2018Q1.txt da historical_data_2018Q1.zip... OK (1196 MB)
  Estraggo historical_data_time_2018Q2.txt da historical_data_2018Q2.zip... OK (1273 MB)
  Estraggo historical_data_time_2018Q3.txt da historical_data_2018Q3.zip... OK (1096 MB)
  Estraggo historical_data_time_2018Q4.txt da historical_data_2018Q4.zip... OK (845 MB)

File TIME estratti: 4
  historical_data_time_2018Q1.txt  (1196 MB)
  historical_data_time_2018Q2.txt  (1273 MB)
  historical_data_time_2018Q3.txt  (1096 MB)
  historical_data_time_2018Q4.txt  (845 MB)


In [19]:
# ============================================================
# CELLA 4 - Carica matched e performance a chunk → Parquet
# ============================================================
# Ottimizzazione RAM: invece di accumulare tutti i chunk in memoria
# e fare un concat finale, scriviamo direttamente su Parquet a chunk.
# Il DataFrame finale è identico a quello del codice originale.

CHUNK_SIZE = 200_000

# --- 4a. Carica il matched (una riga per loan) ---
print(f'Carico matched: {MATCHED_PATH}')
matched = pd.read_csv(MATCHED_PATH, dtype=str, low_memory=False)
print(f'  {len(matched):,} loan x {len(matched.columns)} colonne')

loan_ids = set(matched['loan_sequence_number'].dropna().unique())
print(f'  Loan ID unici: {len(loan_ids):,}')

# --- 4b. Scrivi su Parquet a chunk, senza accumulare in RAM ---
print(f'\nCarico file performance e scrivo su Parquet...')
writer = None
total_rows = 0

for f in perf_txt_files:
    fname = os.path.basename(f)

    with open(f, 'r') as fh:
        first_line = fh.readline().rstrip('\n')
    n_cols_file = len(first_line.split('|'))

    reader = pd.read_csv(
        f, sep='|', header=None,
        names=PERF_COLS_FULL[:n_cols_file],
        dtype=str, low_memory=False,
        chunksize=CHUNK_SIZE,
        on_bad_lines='skip'
    )

    rows_this_file = 0
    for chunk in reader:
        filtered = chunk[chunk['loan_sequence_number'].isin(loan_ids)]
        if filtered.empty:
            continue
        cols_available = [c for c in PERF_COLS_KEEP if c in filtered.columns]
        filtered = filtered[cols_available].copy()

        # Aggiungi colonne mancanti come None per rispettare lo schema fisso
        for col in PERF_COLS_KEEP:
            if col not in filtered.columns:
                filtered[col] = None
        filtered = filtered[PERF_COLS_KEEP]

        table = pa.Table.from_pandas(filtered, schema=PARQUET_SCHEMA, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(PERF_PARQUET, PARQUET_SCHEMA)
        writer.write_table(table)

        rows_this_file += len(filtered)
        total_rows     += len(filtered)
        del filtered, table
        gc.collect()

    print(f'  {fname}: {rows_this_file:,} righe')

if writer:
    writer.close()

print(f'\nPerformance totale  : {total_rows:,} righe')
print(f'Parquet scritto in  : {PERF_PARQUET}')

# Leggi il Parquet — equivalente al pd.concat originale
perf = pd.read_parquet(PERF_PARQUET)
print(f'Loan unici in perf  : {perf["loan_sequence_number"].nunique():,}')

Carico matched: /content/drive/MyDrive/thesis_data/output/matched_2018.csv
  33,585 loan x 98 colonne
  Loan ID unici: 33,585

Carico file performance e scrivo su Parquet...
  historical_data_time_2018Q1.txt: 485,326 righe
  historical_data_time_2018Q2.txt: 390,928 righe
  historical_data_time_2018Q3.txt: 321,488 righe
  historical_data_time_2018Q4.txt: 251,953 righe

Performance totale  : 1,449,695 righe
Parquet scritto in  : /content/perf_tmp.parquet
Loan unici in perf  : 33,585


In [20]:
# ============================================================
# CELLA 5 - Preparazione variabili performance e colonne temporali
# ============================================================

# Conversioni numeriche
for col in ['current_upb', 'loan_age', 'remaining_months_to_maturity',
            'current_interest_rate', 'estimated_ltv', 'current_deferred_upb']:
    if col in perf.columns:
        perf[col] = pd.to_numeric(perf[col], errors='coerce')

# Parsing del periodo YYYYMM
perf['monthly_reporting_period'] = perf['monthly_reporting_period'].str.strip()
perf['period_year']    = perf['monthly_reporting_period'].str[:4].astype(int)
perf['period_month']   = perf['monthly_reporting_period'].str[4:6].astype(int)
perf['quarter']        = ((perf['period_month'] - 1) // 3 + 1).astype(int)
perf['period_quarter'] = perf['period_year'].astype(str) + 'Q' + perf['quarter'].astype(str)

perf_q = perf  # una riga per (loan, mese)

print(f'Osservazioni mensili : {len(perf_q):,}')
print(f'Loan unici           : {perf_q["loan_sequence_number"].nunique():,}')
print(f'Mesi medi per loan   : {len(perf_q) / perf_q["loan_sequence_number"].nunique():.1f}')

Osservazioni mensili : 1,449,695
Loan unici           : 33,585
Mesi medi per loan   : 43.2


In [21]:
# ============================================================
# CELLA 6 - Join con matched (origination + HMDA) e salvataggio
# ============================================================

print('Join performance x origination/HMDA...')

panel = perf_q.merge(
    matched,
    on='loan_sequence_number',
    how='inner'
)

print(f'Panel: {panel.shape[0]:,} righe x {panel.shape[1]} colonne')
print(f'Loan unici nel panel: {panel["loan_sequence_number"].nunique():,}')

# Verifica duplicati su (loan, mese)
dups = panel.duplicated(subset=['loan_sequence_number', 'monthly_reporting_period']).sum()
if dups > 0:
    print(f'ATTENZIONE: {dups:,} righe duplicate su (loan, month).')
else:
    print('OK: nessun duplicato su (loan_sequence_number, month).')

# Salva e libera RAM
print(f'\nSalvataggio in corso: {OUTPUT_PATH}')
panel.to_csv(OUTPUT_PATH, index=False)
size_mb = os.path.getsize(OUTPUT_PATH) / 1e6

print(f'Salvato: {OUTPUT_PATH}')
print(f'Dimensione : {size_mb:.1f} MB')

del panel, perf, perf_q
gc.collect()

print('Completato.')

Join performance x origination/HMDA...
Panel: 1,449,695 righe x 115 colonne
Loan unici nel panel: 33,585
OK: nessun duplicato su (loan_sequence_number, month).

Salvataggio in corso: /content/drive/MyDrive/thesis_data/output/panel_2018.csv
Salvato: /content/drive/MyDrive/thesis_data/output/panel_2018.csv
Dimensione : 785.1 MB
Completato.
